# AI Crypto Hedge Fund — Baseline Single Asset

Offline historical backtesting for **BTC/USDT** (1d). This notebook orchestrates the baseline pipeline implemented in `src/crypto_hf/` — no core business logic lives here.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

from crypto_hf.config import load_config
from crypto_hf.data.loader import load_ohlcv_csv
from crypto_hf.data.validation import validate_ohlcv
from crypto_hf.pipeline.baseline import (
    build_features,
    run_backtests,
    run_baseline_pipeline,
    split_train_test,
)
from crypto_hf.visualization.plots import (
    export_metrics_table,
    plot_drawdown,
    plot_equity_curve,
    plot_metrics_table,
    plot_price_with_sma,
)

plt.style.use("seaborn-v0_8-whitegrid")

## 1. Load config and data

In [2]:
config = load_config(PROJECT_ROOT / "configs" / "baseline.yaml")
raw = load_ohlcv_csv(config.data_path)
print(config)
raw.tail()

symbol='BTC/USDT' timeframe='1d' data_path=PosixPath('/home/gwyllan/projects/ai-crypto-hedge-fund/data/raw/BTC_USDT_1d.csv') initial_cash=10000.0 fee_rate=0.001 train_size=0.7 fast_window=10 slow_window=30 volatility_window=20


,open,high,low,close,volume
timestamp,,,,,
2023-05-11 00:00:00+00:00,33854.205382,34718.865011,33660.704507,34502.889472,3641.856101
2023-05-12 00:00:00+00:00,34532.042166,34880.570977,33140.805223,33256.364883,1967.731889
2023-05-13 00:00:00+00:00,33236.772311,33278.096362,33003.510053,33049.771443,1913.696996
2023-05-14 00:00:00+00:00,33057.293543,33305.516028,31555.683303,31750.378701,2200.949029
2023-05-15 00:00:00+00:00,31769.710587,31838.317399,30751.051327,30816.878939,4934.140530


## 2. Data validation summary

In [ ]:
validation = validate_ohlcv(raw)
print(f"Rows: {validation.row_count}")
print(f"Range: {validation.start.date()} → {validation.end.date()}")
print(f"Columns: {validation.columns}")

## 3. Feature engineering

In [ ]:
features = build_features(raw, config)
features[["close", "returns", "sma_10", "sma_30", "volatility_20", "drawdown"]].tail()

## 4. Train/test split

In [ ]:
train, test = split_train_test(features, config.train_size)
print(f"Train: {len(train)} bars | Test: {len(test)} bars")

## 5. Buy-and-hold benchmark

In [ ]:
results = run_backtests(test, config)
results["buy_and_hold"].metrics

## 6. SMA crossover baseline

In [ ]:
results["sma_crossover"].metrics

## 7. Backtest results

In [ ]:
for name, result in results.items():
    print(name, "final equity:", round(result.equity_curve.iloc[-1], 2))

## 8. Metrics comparison

In [ ]:
metrics_table = export_metrics_table({name: r.metrics for name, r in results.items()})
metrics_table

## 9. Equity curve and drawdown plots

In [ ]:
plot_price_with_sma(test, config.fast_window, config.slow_window)
plot_equity_curve({name: r.equity_curve for name, r in results.items()})
plot_drawdown(results["sma_crossover"].equity_curve)
plot_metrics_table({name: r.metrics for name, r in results.items()})
plt.show()

## 10. Save reports (same as CLI script)

In [ ]:
outputs = run_baseline_pipeline(config, reports_dir=PROJECT_ROOT / "reports")
outputs.metrics_table

## 11. How this baseline can evolve into AI agents

This MVP establishes a reproducible offline loop: **config → data validation → features → signals → backtest → metrics → reports**. Later phases can plug in:

- **ML models** replacing rule-based `signal` generation while keeping the same `BaseStrategy` interface.
- **Econometric models** for regime detection and volatility forecasting as additional features.
- **LLM agents** for narrative/sentiment features and experiment orchestration — still evaluated through the same backtest engine.
- **Portfolio optimization & rebalancing** across many pairs, reusing metrics and visualization layers.

Because positions are shifted by one bar and validation is strict, new alpha sources can be benchmarked fairly against buy-and-hold and SMA crossover.